 *Artificial Intelligence for Vision & NLP* &nbsp; | &nbsp;  *ATU Donegal - Postgrad Diploma in Big Data Analytics & Artificial Intelligence*

# Student Submisison 
Name           : John Ryan         <br>
Student Number : L00007202         <br>
Due Date       : 12th May 2026     <br>
Assignment     : CA2               <br>
Module         : AI for Vision and NLP    <br>
Course         : Postgraduate Diploma in Big Data Analytics and AI

## NLP and Vision Pipeline : High Level
An image of your working pipeline at high level can be inserted here



# Initialisation
Perform pip installs(or use a requirements.txt) <br>
perform imports

## Install packages

In [2]:
# pip installs
# pip install -r requirements.txt

## Imports

In [12]:
# imports
import pandas as pd
import pymupdf
import pytesseract
import os
from PIL import Image
import cv2
import numpy as np
import io
#import fitz
from deskew import determine_skew
from skimage.transform import rotate

# Support Functions

In [4]:
# code here

## NLP

# Define the Corpus

### Apply Confidence Level Threshold

In [13]:
#path to the tesseract program on my laptop
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe" 

#***STAGE 2***
#this additional confidence check is to counter the massive char counts 
def get_confident_text(img, min_confidence=60): #define function to extract text fromimage when confidence >60%
    data = pytesseract.image_to_data(img, config='--psm 6', # use tesseract to get data about image and treat it as a single block
                                     output_type=pytesseract.Output.DATAFRAME) # return the OCR output as a DF
    # Keep only words with confidence above threshold
    data = data[(data['conf'] >= min_confidence) & (data['text'].str.strip() != '')] # only keep the data above the confidence threshold set
    return ' '.join(data['text'].tolist()) # adds text above threshold to the string


### Pre-Processing and Text Extraction

In [16]:
#***STAGE 3***
# - Create function to extract text from JPEGs
def extract_text_from_jpg(path): #define the function to read JPG
    img = Image.open(path) #open JPEG at the path using PIL
    img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)  # using OpenCV to preprocess to improve OCR accuracy, convert to greyscale

    # - Adjust for 90 degree rotation from my scans
    img_cv = cv2.rotate(img_cv, cv2.ROTATE_90_COUNTERCLOCKWISE) # turn everything 90 degrees anti-clockwise

    # - Eliminate any other small skew [PROB OVERKILL- CONSIDER REMOVING]
    angle = determine_skew(img_cv) # calc remaining rotation angle 
    if angle is not None and abs(angle) > 0.5: # initial check to make sure any angle is above 1 degrees 
        img_cv = rotate(img_cv, angle, resize=True, # rotate the image by the angle detected 
                        preserve_range=True).astype(np.uint8) # preserve pixel values 
    # - Reduce Noise
    img_cv = cv2.medianBlur(img_cv, 3) # reduce noise by applying a blur
    
    # - Pixel Variation Handling
    local_var = img_cv.var() # determine the variance
    if local_var > 2500: # setting threshold of 2500 - apply different techniques above and below this threshold
        # where high variance, use Otsu's global thresholdbest suited to this type of image
        _, img_cv = cv2.threshold(img_cv, 0, 255, #convert grayscale to binary image using threshold where everything above goes to white
                        cv2.THRESH_BINARY + cv2.THRESH_OTSU) #Otsu technique calsc the threshold auto rather than setting it manually
    else: 
        img_cv = cv2.adaptiveThreshold(img_cv, 255, # creates binary image
                        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,# uses a weighted Guassian technique to calc thresholdforlower variance images
                        cv2.THRESH_BINARY, 15, 8) # thresholds value based on 15x15 region 

    kernel = np.ones((1,1), np.uint8) # create 1 x 1 kernel/matrix
    img_cv = cv2.morphologyEx(img_cv, cv2.MORPH_OPEN, kernel) # morphological technique to clean up bright specks
    img_cv = cv2.resize(img_cv, None, fx=2, fy=2, # increase the size 2x
                interpolation=cv2.INTER_CUBIC) # apply bicubic interpolation technique when scaling up to improve image quality

    # - Generate String
    return get_confident_text(Image.fromarray(img_cv))  # covert to PIL image for ease of processing and extract text with high conf. score


# - Create function to extract text from PDFs
def extract_text_from_pdf(pdf_path): # Define function to read a PDF
    doc = pymupdf.open(pdf_path) # Open and read PDFs 
    pages = {} #create dictionary for processed text

    # - Extract and label embedded text in PDF pages
    for page_num in range(len(doc)): # loop through the pages in the doc
        page = doc[page_num] #creates object for each page
        page_name = f"{os.path.basename(pdf_path)}_page{page_num + 1}" #create descriptive label for each page
        embedded_text = page.get_text().strip() # extracts embedded text from the PDF - not using OCR
    
    # - Where low character count, use OCR
        if len(embedded_text) > 100: # check that char count >100
            pages[page_name] = embedded_text # then save those pages
        else:
            mat = pymupdf.Matrix(3, 3) # for pages with <100 chars, scale the page 3x resolution
            pix = page.get_pixmap(matrix=mat, colorspace=pymupdf.csGRAY) # render the page as a grayscale image for processeing
            img = Image.open(io.BytesIO(pix.tobytes("png"))) # converts the image/pixmap into an object that PIL can process
            #pages[page_name] = get_confident_text(img) # use OCR on the object and store high-confidence text

            # - Adjust for 90 degree rotation from my scans here too
            img_cv = np.array(img)  # already greyscale, no conversion needed
            img_cv = cv2.rotate(img_cv, cv2.ROTATE_90_COUNTERCLOCKWISE)
            img = Image.fromarray(img_cv)

            pages[page_name] = get_confident_text(img)

    return pages # outputs the 'pages' dict. with the processed results

In [17]:
#***STAGE 4***
# - Create function to check file format before calling other function
def extract_text(path): # define function name
    ext = os.path.splitext(path)[1].lower() # split the path name and change the file ext value to lowercase
    if ext in ['.jpg', '.jpeg', '.png']: # if that ext is one of these values
        return {os.path.basename(path): extract_text_from_jpg(path)} # filename becomes dict. key and OCT extracted text becomes dict. entry
    elif ext == '.pdf': 
        return extract_text_from_pdf(path) # extracts text from PDF - each page has separate key
    else:
        print(f"Unsupported format: {path}") # focus on PDF and JPG for this exercise. Code doesn't cover other formats with certainty 
        return {}

# - Build the corpus 
document_paths = [
    'schoolpros1.pdf',
    'schoolpros2.jpg',
    'schoolpros3.pdf',
    'schoolpros4.jpg',
    'schoolpros5.pdf',
    'schoolpros6.jpg',
    'schoolpros7.pdf',
    'schoolpros8.jpg',
    'schoolpros9.pdf',
    'schoolpros10.jpg',
    'schoolpros11.pdf',
    'schoolpros12.jpg',
    'schoolpros13.pdf',
    # might adde more but that is it for now
]

corpus_raw = {} # create a new dictioanry
for path in document_paths: # loop through the docs in the path
    corpus_raw.update(extract_text(path)) # calls the functions above and merges their output into the corpus dictionary

   
# - Filter out very low text pages
corpus_raw = {k: v for k, v in corpus_raw.items() if len(v.strip()) > 50} # filter out any images or FDF pages with <100 chars
print(f"Corpus size: {len(corpus_raw)} documents") # show how many keys (images/pages) are in the corpus after filtering
for name, text in corpus_raw.items(): # looops through the corpus entries
    print(f"  {name}: {len(text.strip())} characters") # and prints the name of each image/page plus char count


Corpus size: 12 documents
  schoolpros1.pdf_page1: 199 characters
  schoolpros3.pdf_page1: 256 characters
  schoolpros4.jpg: 1544 characters
  schoolpros5.pdf_page1: 867 characters
  schoolpros6.jpg: 519 characters
  schoolpros7.pdf_page1: 880 characters
  schoolpros8.jpg: 210 characters
  schoolpros9.pdf_page1: 1410 characters
  schoolpros10.jpg: 1173 characters
  schoolpros11.pdf_page1: 1045 characters
  schoolpros12.jpg: 241 characters
  schoolpros13.pdf_page1: 186 characters


## Document Text Processing

In [6]:
#!python -m spacy download en_core_web_sm #RAN ONCE ON MAY 3RD, SHOULDNT NEED TO RUN AGAIN

In [18]:
# import modules etc
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import spacy

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

nlp = spacy.load("en_core_web_sm") #load the small English NLP model from spacy
stop_words = set(stopwords.words('english')) #load nltk stopwords
stemmer = PorterStemmer() #load Porter's stemmer

# set up the function to NLP pre-proc, i.e. tokenisation, remove stopswords, stem and lemma the text
def process_document(text): #define the function for pre-proc
    tokens = word_tokenize(text) #tokenise the text breaking into words and punc marks
    tokens = [t.lower() for t in tokens if t.isalpha()] #set to lowercase and remove numbers
    tokens = [t for t in tokens if t not in stop_words] #remove stopwords from tokens
    stemmed = [stemmer.stem(t) for t in tokens] #stem the tokens (porter)
    doc_spacy = nlp(" ".join(tokens)) #rejoin the tokens for lemmatisation
    lemmatised = [token.lemma_.lower() for token in doc_spacy if token.is_alpha] #lemmatise the tokens
    
    return {
        "tokens"    : tokens, #lowercase with stopwords removed
        "stemmed"   : stemmed, #stemmed versions
        "lemmatised": lemmatised, #lemmatised verstions using spacy
        "processed_text": " ".join(lemmatised) #single string of lemmatised words
    }

#Run the process_document function on the docs in the corpus
corpus_processed = {} #create a new dictionary which will hold the processed data
for doc_name, raw_text in corpus_raw.items(): #loop through each doc in the corpus
    print(f"Processing: {doc_name}") #display name of docs processed
    corpus_processed[doc_name] = process_document(raw_text) #apply the pre-proc function and send the output to the new dictionary

print("\nAll documents processed like!") #status message

# ── 6. Build processed text list for TF-IDF ──────────────────────────────────
tfidf_corpus = [corpus_processed[doc]["processed_text"] for doc in corpus_processed] #build list of processed text for each doc, i.e. list with 3 parts
doc_names    = list(corpus_processed.keys()) #save list of doc names / keys

print(f"\nCorpus ready for TF-IDF: {len(tfidf_corpus)} documents") #status message

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jryan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jryan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jryan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Processing: schoolpros1.pdf_page1
Processing: schoolpros3.pdf_page1
Processing: schoolpros4.jpg
Processing: schoolpros5.pdf_page1
Processing: schoolpros6.jpg
Processing: schoolpros7.pdf_page1
Processing: schoolpros8.jpg
Processing: schoolpros9.pdf_page1
Processing: schoolpros10.jpg
Processing: schoolpros11.pdf_page1
Processing: schoolpros12.jpg
Processing: schoolpros13.pdf_page1

All documents processed like!

Corpus ready for TF-IDF: 12 documents


## Tokenisation

### Summary Token Info for the Documents

In [19]:
for doc_name, data in corpus_processed.items(): #loop through the docs in the corpus
    print(f"\n=== {doc_name} ===") #print doc name
    print(f"Token count: {len(data['tokens'])}") #print tokens qty
    print(f"Unique tokens: {len(set(data['tokens']))}") #print unique tokens qty
    print(f"First 5 tokens: {data['tokens'][:5]}") #as example, show first 5 tokens 


=== schoolpros1.pdf_page1 ===
Token count: 16
Unique tokens: 16
First 5 tokens: ['regina', 'mundi', 'college', 'prospectus', 'et']

=== schoolpros3.pdf_page1 ===
Token count: 25
Unique tokens: 23
First 5 tokens: ['contents', 'page', 'message', 'principal', 'mission']

=== schoolpros4.jpg ===
Token count: 169
Unique tokens: 130
First 5 tokens: ['principal', 'essage', 'tro', 'e', 'frincipa']

=== schoolpros5.pdf_page1 ===
Token count: 70
Unique tokens: 65
First 5 tokens: ['mission', 'statement', 'regina', 'mundi', 'college']

=== schoolpros6.jpg ===
Token count: 53
Unique tokens: 40
First 5 tokens: ['ca', 'z', 'é', 'ry', 'x']

=== schoolpros7.pdf_page1 ===
Token count: 74
Unique tokens: 65
First 5 tokens: ['teaching', 'learning', 'highest', 'quality', 'teaching']

=== schoolpros8.jpg ===
Token count: 24
Unique tokens: 23
First 5 tokens: ['e', 'sh', 'mathem', 'nelis', 'r']

=== schoolpros9.pdf_page1 ===
Token count: 126
Unique tokens: 99
First 5 tokens: ['enrichment', 'learning', 'experi

In [20]:
from collections import Counter #import mod to count words

for doc_name, data in corpus_processed.items(): #loop through docs in the corpus
    freq = Counter(data['tokens']) 
    common = freq.most_common(5) #get the 5 most plentiful lemmatised words
    print(f"\nTop words in {doc_name}:") #print the output with a doc name heading
    for word, count in common:
        print(f"  {word}: {count}") #print the lemma and its frequency count


Top words in schoolpros1.pdf_page1:
  regina: 1
  mundi: 1
  college: 1
  prospectus: 1
  et: 1

Top words in schoolpros3.pdf_page1:
  learning: 2
  cycle: 2
  contents: 1
  page: 1
  message: 1

Top words in schoolpros4.jpg:
  e: 21
  school: 4
  principal: 3
  students: 3
  regina: 3

Top words in schoolpros5.pdf_page1:
  school: 3
  aims: 2
  student: 2
  students: 2
  mission: 1

Top words in schoolpros6.jpg:
  e: 9
  q: 3
  z: 2
  x: 2
  c: 2

Top words in schoolpros7.pdf_page1:
  learning: 3
  teaching: 2
  academic: 2
  expertise: 2
  students: 2

Top words in schoolpros8.jpg:
  e: 2
  sh: 1
  mathem: 1
  nelis: 1
  r: 1

Top words in schoolpros9.pdf_page1:
  students: 6
  learning: 5
  activities: 5
  enrichment: 4
  experience: 2

Top words in schoolpros10.jpg:
  school: 4
  pastoral: 3
  care: 3
  students: 3
  well: 3

Top words in schoolpros11.pdf_page1:
  school: 10
  students: 7
  must: 7
  rules: 3
  always: 2

Top words in schoolpros12.jpg:
  e: 2
  yet: 1
  n: 1
  j: 

## Unique Words and Keywords
After Stopwords are Removed

Identify the Words Unique to each Document

In [21]:
doc_words = {name: set(data['tokens']) for name, data in corpus_processed.items()}#creates a new dict comprising sets of tokens

for name, words in doc_words.items(): #loop through the docs to get meaningful words in each doc
    others = set().union(*(doc_words[n] for n in doc_words if n != name)) #create 'others' to be the combo of the docs not being analysed
    unique = words - others # creates unique list by taking words in current doc from combo
    print(f"\nUnique meaningful words in {name}:") 
    print(sorted(unique)[:5]) #prints 5 unique words per doc ordered alphabetically


Unique meaningful words in schoolpros1.pdf_page1:
['courage', 'maintaining', 'nurturing', 'pursuit', 'wisdom']

Unique meaningful words in schoolpros3.pdf_page1:
['contents', 'information', 'message', 'page', 'senior']

Unique meaningful words in schoolpros4.jpg:
['ach', 'achieve', 'agus', 'aim', 'also']

Unique meaningful words in schoolpros5.pdf_page1:
['accommodate', 'based', 'become', 'beliefs', 'board']

Unique meaningful words in schoolpros6.jpg:
['ae', 'bd', 'c', 'ca', 'dj']

Unique meaningful words in schoolpros7.pdf_page1:
['according', 'across', 'approach', 'attainment', 'classes']

Unique meaningful words in schoolpros8.jpg:
['af', 'avariety', 'ce', 'completed', 'follow']

Unique meaningful words in schoolpros9.pdf_page1:
['accordingly', 'adventure', 'art', 'artistic', 'assist']

Unique meaningful words in schoolpros10.jpg:
['able', 'air', 'allow', 'allowing', 'app']

Unique meaningful words in schoolpros11.pdf_page1:
['adhere', 'always', 'appearance', 'appropriate', 'autho

Lexical Diversity per Document - same, DOES THIS ADD ANY VALUE?

In [22]:
for doc_name, data in corpus_processed.items(): #loop through the corpus
    tokens = data['tokens'] #get the tokens
    unique = len(set(tokens)) #unique ones
    total = len(tokens)
    print(f"\n=== {doc_name} ===")
    if total == 0: #provision for images if no tokens found
        print ("Lexical diversity: N/A (no tokens found)") #if none, just print this msg
    else:
        print(f"Lexical diversity: {unique/total*100:.2f}%") #divide one by the other and print result as %



=== schoolpros1.pdf_page1 ===
Lexical diversity: 100.00%

=== schoolpros3.pdf_page1 ===
Lexical diversity: 92.00%

=== schoolpros4.jpg ===
Lexical diversity: 76.92%

=== schoolpros5.pdf_page1 ===
Lexical diversity: 92.86%

=== schoolpros6.jpg ===
Lexical diversity: 75.47%

=== schoolpros7.pdf_page1 ===
Lexical diversity: 87.84%

=== schoolpros8.jpg ===
Lexical diversity: 95.83%

=== schoolpros9.pdf_page1 ===
Lexical diversity: 78.57%

=== schoolpros10.jpg ===
Lexical diversity: 75.00%

=== schoolpros11.pdf_page1 ===
Lexical diversity: 69.00%

=== schoolpros12.jpg ===
Lexical diversity: 92.31%

=== schoolpros13.pdf_page1 ===
Lexical diversity: 82.61%


## Stemming

In [23]:
#from collections import Counter

for doc_name, data in corpus_processed.items(): #loop through
    stems = data['stemmed'] #variable stems
    total = len(stems) #totals stems
    unique = len(set(stems)) #unique stems
    print(f"\n=== {doc_name} ===")
    print(f"Total stems: {total}")
    print(f"Unique stems: {unique}")
    if total == 0: #provision for images if no stems found
        print ("Stemming compression: N/A (no stems found)") #just print this msg
    else:
        print(f"Stemming compression: {(1 - unique/total)*100:.2f}%") #unique as % of total stems



=== schoolpros1.pdf_page1 ===
Total stems: 16
Unique stems: 16
Stemming compression: 0.00%

=== schoolpros3.pdf_page1 ===
Total stems: 25
Unique stems: 23
Stemming compression: 8.00%

=== schoolpros4.jpg ===
Total stems: 169
Unique stems: 127
Stemming compression: 24.85%

=== schoolpros5.pdf_page1 ===
Total stems: 70
Unique stems: 62
Stemming compression: 11.43%

=== schoolpros6.jpg ===
Total stems: 53
Unique stems: 40
Stemming compression: 24.53%

=== schoolpros7.pdf_page1 ===
Total stems: 74
Unique stems: 62
Stemming compression: 16.22%

=== schoolpros8.jpg ===
Total stems: 24
Unique stems: 23
Stemming compression: 4.17%

=== schoolpros9.pdf_page1 ===
Total stems: 126
Unique stems: 89
Stemming compression: 29.37%

=== schoolpros10.jpg ===
Total stems: 112
Unique stems: 79
Stemming compression: 29.46%

=== schoolpros11.pdf_page1 ===
Total stems: 100
Unique stems: 67
Stemming compression: 33.00%

=== schoolpros12.jpg ===
Total stems: 13
Unique stems: 12
Stemming compression: 7.69%

==

Common Stems in the Documents- same, DOES THIS ADD ANY VALUE?

In [24]:
for doc_name, data in corpus_processed.items(): #loop through
    freq = Counter(data['stemmed']) #quantify the stems
    common = freq.most_common(5) #top 5
    print(f"\nTop stems in {doc_name}:") #print results
    for stem, count in common:
        print(f"  {stem}: {count}")



Top stems in schoolpros1.pdf_page1:
  regina: 1
  mundi: 1
  colleg: 1
  prospectu: 1
  et: 1

Top stems in schoolpros3.pdf_page1:
  learn: 2
  cycl: 2
  content: 1
  page: 1
  messag: 1

Top stems in schoolpros4.jpg:
  e: 21
  school: 4
  princip: 3
  student: 3
  regina: 3

Top stems in schoolpros5.pdf_page1:
  student: 4
  school: 3
  aim: 2
  educ: 2
  member: 2

Top stems in schoolpros6.jpg:
  e: 9
  q: 3
  z: 2
  x: 2
  c: 2

Top stems in schoolpros7.pdf_page1:
  learn: 3
  student: 3
  class: 3
  teach: 2
  academ: 2

Top stems in schoolpros8.jpg:
  e: 2
  sh: 1
  mathem: 1
  neli: 1
  r: 1

Top stems in schoolpros9.pdf_page1:
  student: 7
  learn: 5
  activ: 5
  enrich: 4
  experi: 4

Top stems in schoolpros10.jpg:
  student: 5
  school: 4
  pastor: 3
  care: 3
  well: 3

Top stems in schoolpros11.pdf_page1:
  school: 10
  student: 7
  must: 7
  rule: 3
  phone: 3

Top stems in schoolpros12.jpg:
  e: 2
  yet: 1
  n: 1
  j: 1
  se: 1

Top stems in schoolpros13.pdf_page1:
  regi

## Lemmatisation

Lemma Overview per Document - - same, DOES THIS ADD ANY VALUE?

In [25]:
#from collections import Counter

for doc_name, data in corpus_processed.items(): #loop through
    lemmas = data['lemmatised'] #variable lemmas
    total = len(lemmas) #total
    unique = len(set(lemmas)) #unique lemmas
    print(f"\n=== {doc_name} ===")
    print(f"Total lemmas: {total}")
    print(f"Unique lemmas: {unique}")
    if total == 0:
        print ("Lemmatisation compression: N/A (no lemmas found)")
    else:
        print(f"Lemmatisation compression: {(1 - unique/total)*100:.2f}%") #unique as % of total lemmas



=== schoolpros1.pdf_page1 ===
Total lemmas: 16
Unique lemmas: 16
Lemmatisation compression: 0.00%

=== schoolpros3.pdf_page1 ===
Total lemmas: 25
Unique lemmas: 23
Lemmatisation compression: 8.00%

=== schoolpros4.jpg ===
Total lemmas: 169
Unique lemmas: 128
Lemmatisation compression: 24.26%

=== schoolpros5.pdf_page1 ===
Total lemmas: 70
Unique lemmas: 63
Lemmatisation compression: 10.00%

=== schoolpros6.jpg ===
Total lemmas: 54
Unique lemmas: 41
Lemmatisation compression: 24.07%

=== schoolpros7.pdf_page1 ===
Total lemmas: 74
Unique lemmas: 63
Lemmatisation compression: 14.86%

=== schoolpros8.jpg ===
Total lemmas: 24
Unique lemmas: 23
Lemmatisation compression: 4.17%

=== schoolpros9.pdf_page1 ===
Total lemmas: 126
Unique lemmas: 92
Lemmatisation compression: 26.98%

=== schoolpros10.jpg ===
Total lemmas: 112
Unique lemmas: 80
Lemmatisation compression: 28.57%

=== schoolpros11.pdf_page1 ===
Total lemmas: 100
Unique lemmas: 66
Lemmatisation compression: 34.00%

=== schoolpros12.jp

Common Lemmas in the Documents - IT IS MORE MEANINGFUL TO LOOK AT THE DOCS TOGETHER

In [26]:
#OLD - for doc_name, data in corpus_processed.items(): #loop through
#OLD -     freq = Counter(data['lemmatised']) #quantify the stems
    #OLD - common = freq.most_common(5) #top 5
    #OLD - print(f"\nTop lemmas in {doc_name}:") #print results
    #OLD - for lemmas, count in common:
        #OLD - print(f"  {lemmas}: {count}")

all_lemmas = [] # create empty list to hold all lemmas across all documents
for doc_name, data in corpus_processed.items(): # loop through
    all_lemmas.extend(data['lemmatised']) # add each document's lemmas to the combined list

freq = Counter(all_lemmas) # count all lemmas across the entire corpus
common = freq.most_common(10) # top 10
print("Top 10 lemmas in the Prospectus:")
for lemma, count in common:
    print(f"  {lemma}: {count}")

Top 10 lemmas in the Prospectus:
  e: 34
  student: 29
  school: 28
  regina: 12
  mundi: 12
  college: 12
  learn: 11
  parent: 7
  activity: 7
  must: 7


Lemmas Vs Stems - Differences by Token sample

In [27]:
differences = [] #a new list for the differences b/w stems and lemmas

for doc_name, data in corpus_processed.items(): #loop through
    for token, lemma, stem in zip(data['tokens'], data['lemmatised'], data['stemmed']):
        if lemma != stem: #where lemmas and stems don't match
            differences.append((token, lemma, stem)) #add them to differences list

print(f"{'TOKEN':20} {'LEMMA':20} {'STEM':20}") #print spaced header
print("-" * 60)

for token, lemma, stem in differences[:10]: #first 10 results
    print(f"{token:20} {lemma:20} {stem:20}") #print spaced results


TOKEN                LEMMA                STEM                
------------------------------------------------------------
college              college              colleg              
prospectus           prospectus           prospectu           
courage              courage              courag              
commitment           commitment           commit              
excellence           excellence           excel               
nurturing            nurture              nurtur              
community            community            commun              
message              message              messag              
principal            principal            princip             
philosophy           philosophy           philosophi          


In [21]:
#keep this code which creates a function to extract images from PDF
import fitz  # PyMuPDF

def extract_images_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    images = []

    for page_index in range(len(doc)):
        page = doc[page_index]
        image_list = page.get_images(full=True)

        for img in image_list:
            xref = img[0]  # reference to the image object
            base_image = doc.extract_image(xref)
            images.append(base_image)

    return images


In [22]:
# and keep this code applying the function above to the 2 PDFs and saves the images in them to another dictionary-> pdf_images
pdf_files = [
    'CA2 - Application_2026.pdf',
    'ATU Academic Integrity Policy.pdf'
]

pdf_images = {}

for pdf in pdf_files:
    pdf_images[pdf] = extract_images_from_pdf(pdf)


In [26]:
#quantify the number of images detected in the PDFs
for pdf, images in pdf_images.items():
    print(pdf, "→", len(images), "images found")

CA2 - Application_2026.pdf → 8 images found
ATU Academic Integrity Policy.pdf → 13 images found


In [27]:
#determine keys assoc with first image
first_pdf = list(pdf_images.keys())[0]
first_image = pdf_images[first_pdf][0]

print(first_image.keys())

dict_keys(['width', 'height', 'ext', 'colorspace', 'xres', 'yres', 'bpc', 'size', 'image', 'smask', 'cs-name'])


In [28]:
#save the first image to the local drive so it can be opened for inspection
img = first_image["image"]
ext = first_image["ext"]

with open("test_output." + ext, "wb") as f:
    f.write(img)

In [23]:
#run OCR on the four jpg files
import pytesseract
from PIL import Image

image_files = [
    "schoolpros1.jpg",
    "schoolpros2.jpg",
    "schoolpros3.jpg",
    "schoolpros4.jpg"
]

ocr_results = {}

for img_path in image_files:
    text = pytesseract.image_to_string(Image.open(img_path))
    ocr_results[img_path] = text


In [24]:
#print the OCR output
for k, v in ocr_results.items():
    print("-----", k, "-----")
    print(v[:500])   # print first 500 characters


----- schoolpros1.jpg -----
With Wisdom «
ach for All and All for Each

Sas
=
ida cet i aaa
€ ee

----- schoolpros2.jpg -----
Message from the Principal Ms. Yvonne Lucey

As Principal | am immensely proud to lead this exceptional school, working
collaboratively with a dedicated team of staff to support our students and
their parents, in creating a learning environment conducive to excellence.
Choosing a school for your daughter is an extremely important decision and as
parent(s)/guardian(s), you rightly believe that your child deserves the best
education. At Regina Mundi College, we aim to deliver this.

It is our expe
----- schoolpros3.jpg -----
Mission Statement

Regina Mundi College is a voluntary secondary school, founded
by Miss “Daisy” Corrigan in 1961.

It operates under the supervision of a Board of Directors.

The ethos of the school is Christian, based on the philosophy,
official teaching and practice of the Roman Catholic Church,
while respecting other traditions, values and

## TF-IDF
***THIS NEEDS TO BE REDONE BASED ON 3 DOCS IN CORPUS***

from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import matplotlib.pyplot as plt

# ── 1. Prepare your corpus ────────────────────────────────────────────────────
# Each document is one entry in the list
# For now we have one, but you can simply append more later
#corpus = [processed_text]  # processed_text = " ".join(lemmatised_tokens)
corpus = [full_text]  # 
#corpus = [processed_text, processed_text_doc2, processed_text_doc3]

# ── 2. Initialise and fit the TF-IDF Vectorizer ───────────────────────────────
vectorizer = TfidfVectorizer(
    max_features=20,      # top 20 features
    ngram_range=(1, 2)    # single words and bigrams
)

tfidf_matrix = vectorizer.fit_transform(corpus)

# ── 3. Get feature names and scores ──────────────────────────────────────────
feature_names = vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names,
    index=[f"Document {i+1}" for i in range(len(corpus))]
)

print("=== TF-IDF SCORES ===")
print(tfidf_df.round(3))

# ── 4. Top terms ──────────────────────────────────────────────────────────────
top_terms = tfidf_df.T.sort_values("Document 1", ascending=False)

print("\n=== TOP TERMS ===")
print(top_terms)

# ── 5. Visualise ──────────────────────────────────────────────────────────────
plt.figure(figsize=(12, 6))
top_terms["Document 1"].plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Top TF-IDF Terms', fontsize=14)
plt.xlabel('Terms', fontsize=12)
plt.ylabel('TF-IDF Score', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Vision

## Sub Heading 1

In [ ]:
# code here...

# Multi-modal

## Sub Heading 1

In [ ]:
# code here

# Final Output

In [ ]:
# code